In [ ]:
# Load the extension that allows us to compile CUDA code in python notebooks
# Documentation is here: https://nvcc4jupyter.readthedocs.io/en/latest/
!pip install git+https://github.com/andreinechaev/nvcc4jupyter.git
%load_ext nvcc4jupyter

  Cloning https://github.com/andreinechaev/nvcc4jupyter.git to /tmp/pip-req-build-wberk9aj
  Running command git clone --filter=blob:none --quiet https://github.com/andreinechaev/nvcc4jupyter.git /tmp/pip-req-build-wberk9aj
  Resolved https://github.com/andreinechaev/nvcc4jupyter.git to commit 173e636d9a3cb81c6101a160c1c8c86864eaaf5b
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
The nvcc4jupyter extension is already loaded. To reload it, use:
  %reload_ext nvcc4jupyter


In [ ]:
%%cuda_group_save -g "source" -n "data_types.h"
/**
 * A collection of commonly used data types throughout this project.
 */
#pragma once

#include <iostream> // for std::ostream
#include <vector>

namespace csc485b{
namespace a2{

using node_t = int;
using edge_t = int2;

using edge_list_t = std::vector< edge_t >;
using node_list_t = std::vector< node_t >;

} // namespace a2
} // namespace csc485b


In [ ]:
%%cuda_group_save -g "source" -n "cuda_common.h"
/**
 * Standard macros that can be useful for error checking.
 * https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__ERROR.html
 */
#pragma once

#include <cuda.h>

#define CUDA_CALL(exp)                                       \
    do {                                                     \
        cudaError res = (exp);                               \
        if(res != cudaSuccess) {                             \
            printf("Error at %s:%d\n %s\n",                  \
                __FILE__,__LINE__, cudaGetErrorString(res)); \
           exit(EXIT_FAILURE);                               \
        }                                                    \
    } while(0)

#define CHECK_ERROR(msg)                                             \
    do {                                                             \
        cudaError_t err = cudaGetLastError();                        \
        if(cudaSuccess != err) {                                     \
            printf("Error (%s) at %s:%d\n %s\n",                     \
                (msg), __FILE__, __LINE__, cudaGetErrorString(err)); \
            exit(EXIT_FAILURE);                                      \
        }                                                            \
    } while (0)

In [ ]:
%%cuda_group_save -g "source" -n "data_generator.h"
/**
 * Functions for generating random input data with a fixed seed
 */
#pragma once

#include <cassert>  // for assert()
#include <cstddef>  // std::size_t type
#include <random>   // for std::mt19937, std::uniform_int_distribution
#include <vector>

#include "data_types.h"

namespace csc485b {
namespace a2 {

/**
 * Generates and returns a vector of random edges
 * for a graph `G=(V,E)` with `n=|V|=n` and expected `m=|E|`.
 * Referred to as an Erdős-Rényi graph.
 *
 * @see https://networkx.org/documentation/stable/reference/generated/networkx.generators.random_graphs.fast_gnp_random_graph.html#networkx.generators.random_graphs.fast_gnp_random_graph
 */
edge_list_t generate_graph( std::size_t n, std::size_t m )
{
    assert( "At most n(n-1) edges in a simple graph" && m < n * ( n - 1 ) );

    int const probability = ( 100 * m ) / ( n * ( n - 1 ) );

    // for details of random number generation, see:
    // https://en.cppreference.com/w/cpp/numeric/random/uniform_int_distribution
    std::size_t random_seed = 20241008;  // use magic seed
    std::mt19937 rng( random_seed );     // use mersenne twister generator
    std::uniform_int_distribution<> distrib(0, 100);

    edge_list_t random_edges;
    random_edges.reserve( 2 * m );

    for( node_t u = 0; u < n; ++u )
    {
        for( node_t v = u + 1; v < n; ++v )
        {
            auto const dice_roll = distrib( rng );
            if( dice_roll <= probability )
            {
                random_edges.push_back( make_int2( u, v ) );
                random_edges.push_back( make_int2( v, u ) );
            }
        }
    }

    random_edges.resize( random_edges.size() );


    return random_edges;
}

} // namespace a2
} // namespace csc485b


In [ ]:
%%cuda_group_save -g "source" -n "dense_graph.h"
/**
 * The file in which you will implement your DenseGraph GPU solutions!
 */

#include <cstddef>  // std::size_t type

#include "cuda_common.h"
#include "data_types.h"

namespace csc485b {
namespace a2      {

/**
 * A DenseGraph is optimised for a graph in which the number of edges
 * is close to n(n-1). It is represented using an adjacency matrix.
 */
struct DenseGraph
{
  std::size_t n; /**< Number of nodes in the graph. */
  node_t * adjacencyMatrix; /** Pointer to an n x n adj. matrix */

  /** Returns number of cells in the adjacency matrix. */
  __device__ __host__ __forceinline__
  std::size_t matrix_size() const { return n * n; }
};


namespace gpu {


/**
 * Constructs a DenseGraph from an input edge list of m edges.
 *
 * @pre The pointers in DenseGraph g have already been allocated.
 */
__global__
void build_graph(DenseGraph g, edge_t const* edge_list, std::size_t m) {
    std::size_t tid = blockIdx.x * blockDim.x + threadIdx.x;

    if (tid < m) {
        edge_t edge = edge_list[tid];

        std::size_t index = edge.x * g.n + edge.y;

        g.adjacencyMatrix[index] = 1;
    }
}
/**
 * Two hop reachability for DenseGraph using warp primitives
 */
__global__
void two_hop_reachability_warp(DenseGraph g, node_t* scratch) {
    constexpr int WARP_SIZE = 32;

    const std::size_t warp_id = (blockIdx.x * blockDim.x + threadIdx.x) / WARP_SIZE;

    const int lane = threadIdx.x % WARP_SIZE;

    const std::size_t total_elements = g.matrix_size();

    if (warp_id >= total_elements) {
        return;
    }

    const std::size_t row = warp_id / g.n;
    const std::size_t col = warp_id % g.n;

    // ensure diagonal is 0
    if (row == col) {
        if (lane == 0) {
            scratch[warp_id] = 0;
        }
        return;
    }

    bool found = false;

    for (std::size_t i = lane; i < g.n; i += WARP_SIZE) {
        if (g.adjacencyMatrix[row * g.n + i] && g.adjacencyMatrix[i * g.n + col]) {
            found = true;
            break;
        }
    }

    bool result = __any_sync(0xffffffff, found);

    if (lane == 0) {
        scratch[warp_id] = result ? 1 : 0;
    }
}
/**
 * Two hop reachability for DenseGraph using tiling
 */
__global__
void two_hop_reachability(DenseGraph g, node_t* scratch) {
    constexpr int TILE = 16;

    __shared__ node_t tileA[TILE][TILE];
    __shared__ node_t tileB[TILE][TILE];

    const std::size_t row = blockIdx.y * TILE + threadIdx.y;
    const std::size_t col = blockIdx.x * TILE + threadIdx.x;

    node_t sum = 0;

    const std::size_t numTiles = (g.n + TILE - 1) / TILE;

    for (std::size_t t = 0; t < numTiles; ++t) {
        const std::size_t aCol = t * TILE + threadIdx.x;
        const std::size_t bRow = t * TILE + threadIdx.y;

        tileA[threadIdx.y][threadIdx.x] = (row < g.n && aCol < g.n) ? g.adjacencyMatrix[row * g.n + aCol] : 0;
        __syncthreads();
        tileB[threadIdx.y][threadIdx.x] = (bRow < g.n && col < g.n) ? g.adjacencyMatrix[bRow * g.n + col] : 0;

        __syncthreads();

        if (row < g.n && col < g.n) {
            for (int k = 0; k < TILE; ++k) {
                sum += tileA[threadIdx.y][k] * tileB[k][threadIdx.x];
            }
        }
        __syncthreads();
    }

    if (row < g.n && col < g.n) {
        scratch[row * g.n + col] = (row == col) ? 0 : (sum > 0 ? 1 : 0);
    }
}

__global__
void copy_matrix(DenseGraph g, node_t* scratch) {
    std::size_t tid = blockIdx.x * blockDim.x + threadIdx.x;

    if (tid < g.matrix_size()) {
        g.adjacencyMatrix[tid] = scratch[tid];
    }
}


} // namespace gpu
} // namespace a2
} // namespace csc485b

In [ ]:
%%cuda_group_save -g "source" -n "sparse_graph.h"
/**
 * The file in which you will implement your SparseGraph GPU solutions!
 */

#include <cstddef>  // std::size_t type

#include "cuda_common.h"
#include "data_types.h"

namespace csc485b {
namespace a2      {

/**
 * A SparseGraph is optimised for a graph in which the number of edges
 * is close to cn, for a small constanct c. It is represented in CSR format.
 */

struct SparseGraph
{
  std::size_t n; /**< Number of nodes in the graph. */
  std::size_t m; /**< Number of edges in the graph. */
  node_t * neighbours_start_at; /** Pointer to an n=|V| offset array */
  node_t * neighbours; /** Pointer to an m=|E| array of edge destinations */
};


namespace gpu {

/**
 * Constructs a SparseGraph from an input edge list of m edges.
 *
 * @pre The pointers in SparseGraph g have already been allocated.
 */

__global__
void mark_two_hop_neighbors(SparseGraph g, node_t* reachability) {
    std::size_t source = blockIdx.x;

    if (source >= g.n) {
        return;
    }

    std::size_t start = g.neighbours_start_at[source];
    std::size_t end = g.neighbours_start_at[source + 1];

    for (std::size_t i = start + threadIdx.x; i < end; i += blockDim.x) {
        node_t middle = g.neighbours[i];

        std::size_t middle_start = g.neighbours_start_at[middle];
        std::size_t middle_end = g.neighbours_start_at[middle + 1];

        for (std::size_t j = middle_start; j < middle_end; j++) {
            node_t destination = g.neighbours[j];

            if (destination != source) {
                atomicExch(&reachability[source * g.n + destination], 1);
            }
        }
    }
}

__global__
void initialize_offsets(node_t* offsets, std::size_t n) {
    std::size_t i = blockIdx.x * blockDim.x + threadIdx.x;

    if (i <= n) {
        offsets[i] = 0;
    }
}

__global__
void count_edges(SparseGraph g, edge_t const* edge_list, std::size_t m) {
    std::size_t i = blockIdx.x * blockDim.x + threadIdx.x;

    if (i < m) {
        node_t source = edge_list[i].x;
        atomicAdd(&g.neighbours_start_at[source + 1], 1);
    }
}

__global__
void calculate_offsets(node_t* offsets, std::size_t n) {
    if (blockIdx.x == 0 && threadIdx.x == 0) {
        node_t total = 0;

        for (std::size_t i = 0; i < n; i++) {
            node_t degree = offsets[i + 1];
            offsets[i] = total;
            total += degree;
        }

        offsets[n] = total;
    }
}

__global__
void initialize_cursors(node_t const* offsets, node_t* cursors, std::size_t n) {
    std::size_t node = blockIdx.x * blockDim.x + threadIdx.x;

    if (node < n) {
        cursors[node] = offsets[node];
    }
}

__global__
void populate_neighbours(SparseGraph g, edge_t const* edge_list, node_t* cursors, std::size_t m) {
    std::size_t i = blockIdx.x * blockDim.x + threadIdx.x;

    if (i < m) {
        node_t source = edge_list[i].x;
        node_t destination = edge_list[i].y;

        node_t position = atomicAdd(&cursors[source], 1);

        g.neighbours[position] = destination;
    }
}

void build_graph(SparseGraph g, edge_t const* edge_list, std::size_t m) {
    int threads = 256;

    int node_blocks = (g.n + threads) / threads;
    int edge_blocks = (m + threads - 1) / threads;

    node_t* cursors;
    cudaMalloc(&cursors, g.n * sizeof(node_t));

    initialize_offsets<<<node_blocks, threads>>>(g.neighbours_start_at, g.n);

    count_edges<<<edge_blocks, threads>>>(g, edge_list, m);

    calculate_offsets<<<1, 1>>>(g.neighbours_start_at, g.n);

    initialize_cursors<<<node_blocks, threads>>>(g.neighbours_start_at, cursors, g.n);

    populate_neighbours<<<edge_blocks, threads>>>(g, edge_list, cursors, m);

    cudaFree(cursors);
}

__global__ void build_adjacency_bitsets(SparseGraph g, unsigned int* adjacency_bitsets, std::size_t words_per_row) {
    std::size_t source = blockIdx.x;

    if (source >= g.n) {
        return;
    }

    std::size_t start = g.neighbours_start_at[source];
    std::size_t end = g.neighbours_start_at[source + 1];

    for (std::size_t i = start + threadIdx.x; i < end; i += blockDim.x) {
        node_t destination = g.neighbours[i];

        std::size_t word = destination / 32;
        unsigned int mask = 1u << (destination % 32);

        atomicOr(&adjacency_bitsets[source * words_per_row + word], mask);
    }
}

__global__ void calculate_two_hop_bitsets(SparseGraph g, const unsigned int* adjacency_bitsets, unsigned int* reachability_bitsets, std::size_t words_per_row) {
    std::size_t index = blockIdx.x * blockDim.x + threadIdx.x;

    std::size_t total_words = g.n * words_per_row;

    if (index >= total_words) {
        return;
    }

    std::size_t source = index / words_per_row;
    std::size_t word = index % words_per_row;

    std::size_t start = g.neighbours_start_at[source];
    std::size_t end = g.neighbours_start_at[source + 1];

    unsigned int result = 0;

    for (std::size_t i = start; i < end; i++) {
        node_t middle = g.neighbours[i];

        result |= adjacency_bitsets[static_cast<std::size_t>(middle) * words_per_row + word];
    }

    if (source / 32 == word) {
        result &= ~(1u << (source % 32));
    }

    reachability_bitsets[index] = result;
}

__global__ void count_two_hop_neighbors(const unsigned int* reachability_bitsets, node_t* counts, std::size_t n, std::size_t words_per_row) {
    std::size_t source = blockIdx.x * blockDim.x + threadIdx.x;

    if (source >= n) {
        return;
    }

    node_t count = 0;

    for (std::size_t word = 0; word < words_per_row; word++) {
        unsigned int bits = reachability_bitsets[source * words_per_row + word];
        count += __popc(bits);
    }

    counts[source] = count;
}

__global__ void calculate_two_hop_offsets(node_t* output_offsets, const node_t* counts, std::size_t n) {
    if (blockIdx.x == 0 && threadIdx.x == 0) {
        output_offsets[0] = 0;

        for (std::size_t i = 0; i < n; i++) {
            output_offsets[i + 1] = output_offsets[i] + counts[i];
        }
    }
}

__global__ void populate_two_hop_neighbours(const unsigned int* reachability_bitsets, node_t* output_offsets,
                                            node_t* output_neighbours, std::size_t n, std::size_t words_per_row) {

    std::size_t source = blockIdx.x * blockDim.x + threadIdx.x;

    if (source >= n) {
        return;
    }

    std::size_t output_position = output_offsets[source];

    for (std::size_t word = 0; word < words_per_row; word++) {
        unsigned int bits = reachability_bitsets[source * words_per_row + word];

        while (bits != 0) {
            unsigned int bit = __ffs(bits) - 1;

            node_t destination = static_cast<node_t>(word * 32 + bit);

            if (destination < n) {
                output_neighbours[output_position] = destination;
                output_position++;
            }

            bits &= (bits - 1);
        }
    }
}

void two_hop_reachability(SparseGraph g, node_t* output_offsets, node_t* output_neighbours) {
    if (g.n == 0) {
        return;
    }

    const int threads = 256;
    const std::size_t words_per_row = (g.n + 31) / 32;
    const std::size_t total_words = g.n * words_per_row;
    unsigned int* adjacency_bitsets;
    unsigned int* reachability_bitsets;
    node_t* counts;

    cudaMalloc((void**)&adjacency_bitsets, total_words * sizeof(unsigned int));
    cudaMalloc((void**)&reachability_bitsets, total_words * sizeof(unsigned int));
    cudaMalloc((void**)&counts, g.n * sizeof(node_t));

    cudaMemset(adjacency_bitsets, 0, total_words * sizeof(unsigned int));

    build_adjacency_bitsets<<<g.n, threads>>>(g, adjacency_bitsets, words_per_row);

    const std::size_t total_threads = total_words;

    int reachability_blocks = static_cast<int>((total_threads + threads - 1) / threads);

    calculate_two_hop_bitsets<<<reachability_blocks, threads>>>(g, adjacency_bitsets, reachability_bitsets, words_per_row);

    int node_blocks = static_cast<int>((g.n + threads - 1) / threads);

    count_two_hop_neighbors<<<node_blocks, threads>>>(reachability_bitsets, counts, g.n, words_per_row);

    calculate_two_hop_offsets<<<1, 1>>>(output_offsets, counts, g.n);

    populate_two_hop_neighbours<<<node_blocks, threads>>>(reachability_bitsets, output_offsets, output_neighbours, g.n, words_per_row);

    cudaFree(adjacency_bitsets);
    cudaFree(reachability_bitsets);
    cudaFree(counts);
}



} // namespace gpu
} // namespace a2
} // namespace csc485b

In [ ]:
%%cuda_group_save -g "source" -n "main.cu"
/**
 * Driver for the benchmark comparison. Generates random data,
 * runs the CPU baseline, and then runs your code.
 */

#include <chrono>   // for timing
#include <iostream> // std::cout, std::endl
#include <iterator> // std::ostream_iterator
#include <vector>

#include "dense_graph.h"
#include "sparse_graph.h"

#include "data_generator.h"
#include "data_types.h"


//removed run because i wouldve had to add checks for if the graph was sparse

// CPU based two hop to check if two-hop on gpu is returning the correct result.
bool two_hop_comparison(int n, csc485b::a2::node_t *input, csc485b::a2::node_t *gpu_result) {
    bool matches = true;
    for (int row=0; row < n; row++) {
        for (int col = 0; col < n ; col++) {
            bool reach_in_two = false;

            for (int k = 0; k < n ; k++) {
                if (input[row * n + k] != 0 && input[k * n + col] != 0) {
                    reach_in_two = true;
                    break;
                }
            }
            csc485b::a2::node_t expected = (row == col || !reach_in_two) ? 0 : 1;
            csc485b::a2::node_t actual =  gpu_result[row * n + col ];
            if (actual != expected) {
                matches = false;
                //std::cout<< "\n Not same result: row" << row << " ,col " << col << ", expected" << expected << ", actual" << actual << "\n";
            }
        }
    }
    if (matches) {
        std::cout <<"\nCORRECT GPU RESULT MATCHES CPU\n";
    }
    return matches;
}

/**
 * Allocates space for a dense graph and then runs the test code on it.
 */
void run_dense( csc485b::a2::edge_t const * d_edges, std::size_t n, std::size_t m ) {
    using namespace csc485b;

    // allocate device DenseGraph
    a2::node_t * d_matrix;
    cudaMalloc( (void**)&d_matrix, sizeof( a2::node_t ) * n * n );
    cudaMemset(d_matrix, 0, sizeof(a2::node_t) * n * n);
    a2::DenseGraph d_dg{ n, d_matrix };

    cudaDeviceSynchronize();
    auto const build_start = std::chrono::high_resolution_clock::now();

    int threads = 256;
    int blocks = (n + threads - 1) / threads;
    csc485b::a2::gpu::build_graph<<<blocks, threads>>>(d_dg, d_edges, m);

    cudaDeviceSynchronize();

    csc485b::a2::node_t* scratch;
    cudaMalloc((void**)&scratch, sizeof(csc485b::a2::node_t) * d_dg.n * d_dg.n);

    dim3 reachabilityThreads(16, 16);
    dim3 reachabilityBlocks((d_dg.n + 15) / 16, (d_dg.n + 15) / 16);

    dim3 warp_block(threads);

    dim3 warp_grid((d_dg.matrix_size() * 32 + threads - 1) / threads);

    auto const reachability_start = std::chrono::high_resolution_clock::now();

    //csc485b::a2::gpu::two_hop_reachability_warp<<<warp_grid, warp_block>>>(d_dg, scratch);
    csc485b::a2::gpu::two_hop_reachability<<<reachabilityBlocks, reachabilityThreads>>>(d_dg, scratch);

    cudaDeviceSynchronize();
    auto const end = std::chrono::high_resolution_clock::now();

    int copyThreads = 256;
    int copyBlocks = (d_dg.matrix_size() + copyThreads - 1) / copyThreads;
    csc485b::a2::gpu::copy_matrix<<<copyBlocks, copyThreads>>>(d_dg, scratch);

    cudaDeviceSynchronize();
    cudaFree(scratch);

    std::cout << "Build time: "
              << std::chrono::duration_cast<std::chrono::microseconds>(reachability_start - build_start).count()
              << " us"
              << std::endl;

    std::cout << "Reachability time: "
              << std::chrono::duration_cast<std::chrono::microseconds>(end - reachability_start).count()
              << " us"
              << std::endl;

    // check output?
    std::vector< a2::node_t > host_matrix( d_dg.matrix_size() );
    a2::DenseGraph dg{ n, host_matrix.data() };
    cudaMemcpy( dg.adjacencyMatrix, d_dg.adjacencyMatrix, sizeof( a2::node_t ) * d_dg.matrix_size(), cudaMemcpyDeviceToHost );
    std::copy( host_matrix.cbegin(), host_matrix.cend(), std::ostream_iterator< a2::node_t >( std::cout, " " ) );


    std::vector<a2::edge_t> host_edges(m);
    cudaMemcpy(host_edges.data(), d_edges, sizeof(a2::edge_t) * m, cudaMemcpyDeviceToHost);
    std::vector<a2::node_t> input_matrix( d_dg.matrix_size(), 0 );
    for (auto& edge : host_edges) {
        input_matrix[edge.x * n + edge.y] = 1;
    }
    two_hop_comparison(n, input_matrix.data(), host_matrix.data());

    // clean up
    cudaFree( d_matrix );
}

/**
 * Allocates space for a sparse graph and then runs the test code on it.
 */
void run_sparse( csc485b::a2::edge_t const * d_edges, std::size_t n, std::size_t m ) {
    using namespace csc485b;

    // allocate device SparseGraph
    a2::node_t * d_offsets, * d_neighbours;
    cudaMalloc( (void**)&d_offsets,    sizeof(a2::node_t) * (n + 1) );
    cudaMalloc( (void**)&d_neighbours, sizeof(a2::node_t) * n * n );
    a2::SparseGraph d_sg{ n, m, d_offsets, d_neighbours };

    int threads = 256;
    int blocks = (d_sg.n + threads - 1) / threads;

    cudaDeviceSynchronize();
    auto const build_start = std::chrono::high_resolution_clock::now();

    csc485b::a2::gpu::build_graph(d_sg, d_edges, m);

    cudaDeviceSynchronize();

    csc485b::a2::node_t* output_offsets;
    csc485b::a2::node_t* output_neighbours;

    cudaMalloc((void**)&output_offsets, sizeof(csc485b::a2::node_t) * (d_sg.n + 1));

    cudaMalloc((void**)&output_neighbours, sizeof(csc485b::a2::node_t) * d_sg.n * d_sg.n);

    // Run sparse two-hop reachability.
    int reachabilityThreads = 256;
    int reachabilityBlocks = (d_sg.n + reachabilityThreads - 1) / reachabilityThreads;

    auto const reachability_start = std::chrono::high_resolution_clock::now();

    csc485b::a2::gpu::two_hop_reachability(d_sg, output_offsets, output_neighbours);

    cudaDeviceSynchronize();
    auto const end = std::chrono::high_resolution_clock::now();

    cudaMemcpy(d_sg.neighbours_start_at, output_offsets, sizeof(csc485b::a2::node_t) * (d_sg.n + 1), cudaMemcpyDeviceToDevice);

    cudaMemcpy(d_sg.neighbours, output_neighbours, sizeof(csc485b::a2::node_t) * d_sg.n * d_sg.n, cudaMemcpyDeviceToDevice);

    cudaDeviceSynchronize();

    std::vector<a2::node_t> host_offsets(n + 1);

    cudaMemcpy(host_offsets.data(), d_offsets, sizeof(a2::node_t) * (n + 1), cudaMemcpyDeviceToHost);

    std::cout << "Build time: "
              << std::chrono::duration_cast<std::chrono::microseconds>(reachability_start - build_start).count()
              << " us"
              << std::endl;

    std::cout << "Reachability time: "
              << std::chrono::duration_cast<std::chrono::microseconds>(end - reachability_start).count()
              << " us"
              << std::endl;

    std::size_t output_m = host_offsets[n];
    std::vector<a2::node_t> host_neighbours(output_m);

    cudaMemcpy(host_neighbours.data(), d_neighbours, sizeof(a2::node_t) * output_m, cudaMemcpyDeviceToHost);

    std::cout << "\nTwo-hop CSR Offsets: ";

    for (auto offset : host_offsets) {
        std::cout << offset << " ";
    }

    std::cout << "\nTwo-hop CSR Neighbours: ";

    for (auto neighbour : host_neighbours) {
        std::cout << neighbour << " ";
    }

    std::cout << "\nTotal two-hop neighbours: " << output_m << "\n";

    cudaFree(output_offsets);
    cudaFree(output_neighbours);
    cudaFree( d_neighbours );
    cudaFree( d_offsets );
}


int main() {
    using namespace csc485b;

    // Create input
    std::size_t constexpr n = 4096;
    std::size_t constexpr expected_degree = n >> 1;

    a2::edge_list_t const graph = a2::generate_graph( n, n * expected_degree );

    std::size_t const m = graph.size();

    // lazily echo out input graph
    for( auto const& e : graph )
    {
        std::cout << "(" << e.x << "," << e.y << ") ";
    }

    // allocate and memcpy input to device
    a2::edge_t * d_edges;
    cudaMalloc( (void**)&d_edges, sizeof( a2::edge_t ) * m );
    cudaMemcpyAsync( d_edges, graph.data(), sizeof( a2::edge_t ) * m, cudaMemcpyHostToDevice );

    // run your code!
    run_dense ( d_edges, n, m );
    run_sparse( d_edges, n, m );

    return EXIT_SUCCESS;
}

In [ ]:
%cuda_group_run --group "source" --compiler-args "-O3 -g -std=c++20 -arch=sm_75"

Buffered data was truncated after reaching the output size limit.